# EDA: EEGMMIDB motor imagery

Exploratory only — nothing here is part of the reproducible pipeline (that lives in `src/mibci`). The goal is to *see* the signal: raw traces, the mu/beta spectral peak, the event structure, and the sensorimotor lateralisation that the classifiers will exploit.

Run with the project venv kernel (`.venv`).

In [ ]:
import mne
from mne.datasets import eegbci

# One subject, the imagined left/right fist runs.
SUBJECT, RUNS = 1, [4, 8, 12]
paths = eegbci.load_data(SUBJECT, RUNS, update_path=True, verbose='ERROR')
raw = mne.concatenate_raws([mne.io.read_raw_edf(p, preload=True, verbose='ERROR') for p in paths])
eegbci.standardize(raw)
raw.set_montage('standard_1005', on_missing='warn')
print(raw)

## 1. Raw traces and the event annotations
T1 = imagined left fist, T2 = imagined right fist, T0 = rest.

In [ ]:
print('channels:', len(raw.ch_names), '| sfreq:', raw.info['sfreq'])
print('annotation counts:')
import collections
print(collections.Counter(raw.annotations.description))
raw.plot(duration=10, n_channels=20);

## 2. Power spectrum — find the mu/beta band
Expect a peak around 10 Hz (mu) over central electrodes; this is why the pipeline band-passes 8–30 Hz.

In [ ]:
raw.compute_psd(fmin=1, fmax=45).plot();

## 3. Sensorimotor lateralisation (the thing we decode)
Band-pass to mu/beta, epoch on the cues, and compare band power for left vs right imagery. We expect contralateral desynchronisation: left-fist imagery weaker over C4, right-fist weaker over C3.

In [ ]:
raw.filter(8, 30, fir_design='firwin', verbose='ERROR')
events, event_id = mne.events_from_annotations(raw, verbose='ERROR')
# T1/T2 are the imagery cues; codes depend on annotation order.
epochs = mne.Epochs(raw, events, event_id={k: v for k, v in event_id.items() if k in ('T1', 'T2')},
                    tmin=0.5, tmax=2.5, baseline=None, preload=True, picks='eeg', verbose='ERROR')
# Band power = variance per channel; topomap per class.
import numpy as np
for name in ('T1', 'T2'):
    power = epochs[name].get_data().var(axis=2).mean(axis=0)
    mne.viz.plot_topomap(power, epochs.info, show=True)
    print(name, 'plotted')